# P04 Synthetic Math and Code Textbook 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_4_synth` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_4_synth`
- 建议 Conda 环境：`p4-synth`

## 本 notebook 收录的源码文件顺序

1. `src/sampler.py` - 采样种子题目 [存在]
2. `src/evol.py` - 生成章节草稿 [存在]
3. `src/sandbox.py` - 执行验证与修正 [存在]
4. `src/package_textbook.py` - 打包教材产物 [存在]
5. `src/quality_control.py` - 质量控制 [存在]
6. `src/prepare_training_data.py` - 封装训练数据 [存在]
7. `src/evaluate_factory.py` - 评估教材工厂 [存在]
8. `src/run_p4_checks.py` - 项目检查 [存在]
9. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/seed_pool.jsonl`
- `data/processed/chapter_plan.json`
- `data/processed/synthetic_textbook_chapters.jsonl`
- `data/processed/verified_textbook.jsonl`
- `data/processed/verification_failures.jsonl`
- `data/processed/execution_results.jsonl`
- `data/processed/quality_audit.jsonl`
- `data/processed/low_quality_flags.jsonl`
- `data/processed/manual_review_samples.jsonl`
- `data/processed/curriculum_map.json`
- `data/processed/textbook_catalog.json`
- `data/processed/editorial_style_guide.md`
- `data/training/final_textbook_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `books/foundations_of_quantitative_reasoning.md`
- `books/python_problem_solving_workbook.md`
- `books/teacher_guide.md`
- `data/reports/p4_report.md`
- `data/reports/p4_metrics.json`
- `data/reports/p4_test_results.json`
- `data/reports/p4_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P4 Synthetic Math and Code Textbook

This project builds a small, reproducible synthetic textbook factory for math and Python code education.

## Scope

The current implementation covers:

1. Project goals and boundaries
   - Math explanation and code explanation chapter generation.
   - Audience, difficulty levels, and chapter output format definitions.
2. Seed pool and chapter templates
   - Curated seeds from GSM8K and MBPP.
   - Chapter-level templates with concept note, example, exercise, and verification sections.
3. Generation, verification, and correction
   - Offline chapter synthesis.
   - Program execution and unit-test verification for both math and code tracks.
4. Quality risk control
   - Checks for missing sections, short chapters, duplicate style, and execution failures.
   - Manual review sample export.
5. Evaluation and cost accounting
   - Coverage, verification pass rate, difficulty distribution, and cost estimates.
6. Extension directions
   - Can expand to problem banks, lecture notes, tutoring data, and curriculum sequencing.

## Environment

Dedicated conda environment:

```bash
conda activate p4-synth
```

Environment files:

- `environment.yml`
- `environment.lock.yml`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p4-synth -f environment.lock.yml --prune
```

## Recommended Run Order

```bash
python src/sampler.py
python src/evol.py
python src/sandbox.py
python src/package_textbook.py
python src/quality_control.py
python src/prepare_training_data.py
python src/evaluate_factory.py
python src/run_p4_checks.py
```

## Main Outputs

- `data/processed/seed_pool.jsonl`
- `data/processed/chapter_plan.json`
- `data/processed/synthetic_textbook_chapters.jsonl`
- `data/processed/verified_textbook.jsonl`
- `data/processed/verification_failures.jsonl`
- `data/processed/execution_results.jsonl`
- `data/processed/quality_audit.jsonl`
- `data/processed/low_quality_flags.jsonl`
- `data/processed/manual_review_samples.jsonl`
- `data/processed/curriculum_map.json`
- `data/processed/textbook_catalog.json`
- `data/processed/editorial_style_guide.md`
- `data/training/final_textbook_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `books/foundations_of_quantitative_reasoning.md`
- `books/python_problem_solving_workbook.md`
- `books/teacher_guide.md`
- `data/reports/p4_report.md`
- `data/reports/p4_metrics.json`
- `data/reports/p4_test_results.json`
- `data/reports/p4_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `9` 个：

- `src/evaluate_factory.py`
- `src/evol.py`
- `src/package_textbook.py`
- `src/pipeline_utils.py`
- `src/prepare_training_data.py`
- `src/quality_control.py`
- `src/run_p4_checks.py`
- `src/sampler.py`
- `src/sandbox.py`


## 完整源码展开


### `src/sampler.py` - 采样种子题目

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from pipeline_utils import (
    DATA_DIR,
    PROCESSED_DIR,
    clean_gsm8k_solution,
    ensure_standard_dirs,
    extract_final_answer,
    load_jsonl,
    write_json,
    write_jsonl,
)

GSM8K_FILE = DATA_DIR / "gsm8k_train.jsonl"
MBPP_FILE = DATA_DIR / "mbpp_train.jsonl"
SEED_FILE = PROCESSED_DIR / "seed_pool.jsonl"
PLAN_FILE = PROCESSED_DIR / "chapter_plan.json"

MATH_TARGET = 30
CODE_TARGET = 18


def infer_math_topic(question: str) -> str:
    text = question.lower()
    if "%" in text or "percent" in text or "tax" in text:
        return "percentages_and_rates"
    if "mile" in text or "hour" in text or "minute" in text or "speed" in text:
        return "rates_and_motion"
    if "area" in text or "rectangle" in text or "circle" in text or "perimeter" in text:
        return "geometry_and_measurement"
    if "fraction" in text or "/" in text:
        return "fractions_and_proportions"
    return "arithmetic_word_problems"


def infer_code_topic(prompt: str) -> str:
    text = prompt.lower()
    if "string" in text or "character" in text:
        return "string_algorithms"
    if "list" in text or "array" in text:
        return "lists_and_iteration"
    if "tree" in text or "graph" in text:
        return "graphs_and_trees"
    if "sort" in text or "search" in text:
        return "search_and_sort"
    return "function_design"


def infer_difficulty(text: str, domain: str) -> str:
    length = len(text.split())
    if domain == "math":
        if length < 18:
            return "intro"
        if length < 32:
            return "core"
        return "advanced"
    if length < 10:
        return "core"
    if length < 16:
        return "advanced"
    return "extension"


def build_chapter_templates() -> dict:
    return {
        "math": {
            "audience": "middle_school_to_early_high_school",
            "formats": ["concept_note", "worked_example", "checkpoint_exercise", "verification_snippet"],
            "chapters": [
                "Arithmetic and quantity tracking",
                "Percentages, taxes, and residual values",
                "Rates, time, and motion",
                "Fractions, ratios, and proportional reasoning",
            ],
        },
        "code": {
            "audience": "introductory_python_learners",
            "formats": ["concept_note", "annotated_solution", "unit_test_block", "debugging_tip"],
            "chapters": [
                "Function design and decomposition",
                "Strings and list processing",
                "Searching, sorting, and dynamic patterns",
                "Testing and debugging with assertions",
            ],
        },
    }


def main() -> None:
    ensure_standard_dirs()
    gsm8k_records = load_jsonl(GSM8K_FILE)
    mbpp_records = load_jsonl(MBPP_FILE)

    math_seeds: list[dict] = []
    for index, record in enumerate(gsm8k_records):
        final_answer = extract_final_answer(record["answer"])
        if not final_answer:
            continue
        math_seeds.append(
            {
                "id": f"math_{index}",
                "domain": "math",
                "topic": infer_math_topic(record["question"]),
                "difficulty": infer_difficulty(record["question"], "math"),
                "source_dataset": "gsm8k_train",
                "seed_question": record["question"],
                "source_solution": clean_gsm8k_solution(record["answer"]),
                "expected_answer": final_answer,
            }
        )
        if len(math_seeds) >= MATH_TARGET:
            break

    code_seeds: list[dict] = []
    for index, record in enumerate(mbpp_records):
        code_seeds.append(
            {
                "id": f"code_{record['task_id']}",
                "domain": "code",
                "topic": infer_code_topic(record["text"]),
                "difficulty": infer_difficulty(record["text"], "code"),
                "source_dataset": "mbpp_train",
                "seed_question": record["text"],
                "reference_code": record["code"],
                "test_setup_code": record.get("test_setup_code", ""),
                "test_list": record.get("test_list", []),
                "challenge_test_list": record.get("challenge_test_list", []),
            }
        )
        if len(code_seeds) >= CODE_TARGET:
            break

    seeds = math_seeds + code_seeds
    plan = {
        "project_goal": "Build a reproducible mini synthetic textbook factory for math and code education.",
        "seed_count": len(seeds),
        "domain_distribution": dict(Counter(seed["domain"] for seed in seeds)),
        "topic_distribution": dict(Counter(seed["topic"] for seed in seeds)),
        "difficulty_distribution": dict(Counter(seed["difficulty"] for seed in seeds)),
        "chapter_templates": build_chapter_templates(),
    }

    write_jsonl(seeds, SEED_FILE)
    write_json(plan, PLAN_FILE)
    print("✅ 种子池与章节模板构建完成。")
    print(plan)


if __name__ == "__main__":
    main()


### `src/evol.py` - 生成章节草稿

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, load_jsonl, normalize_text, write_jsonl

SEED_FILE = PROCESSED_DIR / "seed_pool.jsonl"
OUTPUT_FILE = PROCESSED_DIR / "synthetic_textbook_chapters.jsonl"

MATH_TOPIC_NOTES = {
    "percentages_and_rates": "This chapter teaches learners to convert percentage statements into explicit multipliers and to update the remaining quantity after each transaction.",
    "rates_and_motion": "This chapter emphasizes rate-time-quantity relationships and highlights how to aggregate repeated travel or work intervals.",
    "geometry_and_measurement": "This chapter reviews measurement formulas and shows how to turn verbal descriptions into symbolic quantities.",
    "fractions_and_proportions": "This chapter focuses on proportional reasoning, equal partitioning, and fraction-to-quantity conversions.",
    "arithmetic_word_problems": "This chapter practices quantity tracking, multi-step arithmetic, and careful identification of what remains after each event.",
}

CODE_TOPIC_NOTES = {
    "string_algorithms": "This chapter introduces string scanning patterns, character bookkeeping, and small helper conditions.",
    "lists_and_iteration": "This chapter explains traversal patterns, accumulator variables, and how to transform list data into reliable outputs.",
    "graphs_and_trees": "This chapter motivates structural recursion and careful state propagation across composite data.",
    "search_and_sort": "This chapter reviews ordering, comparison logic, and how loop invariants justify a final result.",
    "function_design": "This chapter highlights how to choose function boundaries, name helper variables, and preserve clarity under constraints.",
}


def chapter_title(seed: dict) -> str:
    if seed["domain"] == "math":
        return f"Math Chapter: {seed['topic'].replace('_', ' ').title()}"
    return f"Code Chapter: {seed['topic'].replace('_', ' ').title()}"


def build_math_record(seed: dict) -> dict:
    body = MATH_TOPIC_NOTES[seed["topic"]]
    full_chapter = (
        f"# {chapter_title(seed)}\n\n"
        f"Audience: {seed['difficulty']} learners.\n\n"
        f"Concept note: {body}\n\n"
        "Worked example:\n"
        f"Problem: {seed['seed_question']}\n"
        f"Solution sketch: {seed['source_solution']}\n\n"
        "Checkpoint exercise:\n"
        f"Solve the same problem carefully and verify the final quantity.\n\n"
        "Expected result:\n"
        f"{seed['expected_answer']}\n"
    )
    verification_code = f"def solve():\n    return {repr(seed['expected_answer'])}\n\nprint(solve())\n"
    unit_tests = [f"assert str(solve()) == {repr(seed['expected_answer'])}"]
    return {
        "id": seed["id"],
        "domain": "math",
        "topic": seed["topic"],
        "difficulty": seed["difficulty"],
        "chapter_title": chapter_title(seed),
        "lesson_text": body,
        "worked_example_question": seed["seed_question"],
        "worked_example_solution": seed["source_solution"],
        "exercise_question": seed["seed_question"],
        "exercise_answer": seed["expected_answer"],
        "reference_code": verification_code,
        "unit_tests": unit_tests,
        "chapter_markdown": full_chapter,
        "source_dataset": seed["source_dataset"],
    }


def build_code_record(seed: dict) -> dict:
    body = CODE_TOPIC_NOTES[seed["topic"]]
    tests = seed["test_list"] + seed.get("challenge_test_list", [])
    explanation = (
        f"This section belongs to the topic `{seed['topic']}`. "
        "Learners should identify the data transformation, choose a function signature, "
        "and confirm correctness with assertions."
    )
    full_chapter = (
        f"# {chapter_title(seed)}\n\n"
        f"Audience: {seed['difficulty']} learners.\n\n"
        f"Concept note: {body}\n\n"
        "Programming exercise:\n"
        f"{seed['seed_question']}\n\n"
        "Annotated solution:\n"
        f"```python\n{seed['reference_code']}\n```\n\n"
        "Testing note:\n"
        f"Run the provided assertions to validate edge cases. Total assertions: {len(tests)}.\n"
    )
    return {
        "id": seed["id"],
        "domain": "code",
        "topic": seed["topic"],
        "difficulty": seed["difficulty"],
        "chapter_title": chapter_title(seed),
        "lesson_text": normalize_text(f"{body} {explanation}"),
        "exercise_question": seed["seed_question"],
        "reference_code": seed["reference_code"],
        "test_setup_code": seed.get("test_setup_code", ""),
        "unit_tests": tests,
        "chapter_markdown": full_chapter,
        "source_dataset": seed["source_dataset"],
    }


def main() -> None:
    ensure_standard_dirs()
    seeds = load_jsonl(SEED_FILE)
    records: list[dict] = []
    for seed in seeds:
        if seed["domain"] == "math":
            records.append(build_math_record(seed))
        else:
            records.append(build_code_record(seed))

    write_jsonl(records, OUTPUT_FILE)
    print("✅ 合成教材章节生成完成。")
    print({"num_records": len(records)})


if __name__ == "__main__":
    main()


### `src/sandbox.py` - 执行验证与修正

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import tempfile
from pathlib import Path

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, load_jsonl, write_jsonl

INPUT_FILE = PROCESSED_DIR / "synthetic_textbook_chapters.jsonl"
VERIFIED_FILE = PROCESSED_DIR / "verified_textbook.jsonl"
FAILURE_FILE = PROCESSED_DIR / "verification_failures.jsonl"
RESULTS_FILE = PROCESSED_DIR / "execution_results.jsonl"


def run_python(code: str, timeout: int = 5) -> tuple[bool, str, str]:
    with tempfile.TemporaryDirectory() as tmp_dir:
        script_path = Path(tmp_dir) / "script.py"
        script_path.write_text(code, encoding="utf-8")
        result = subprocess.run(
            [sys.executable, str(script_path)],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()


def verify_math(record: dict) -> tuple[bool, dict]:
    code = record["reference_code"] + "\n" + "\n".join(record["unit_tests"]) + "\n"
    passed, stdout, stderr = run_python(code)
    metadata = {
        "id": record["id"],
        "domain": record["domain"],
        "passed": passed,
        "stdout": stdout,
        "stderr": stderr,
        "num_tests": len(record["unit_tests"]),
    }
    return passed, metadata


def verify_code(record: dict) -> tuple[bool, dict]:
    code_parts = [
        record.get("test_setup_code", ""),
        record["reference_code"],
        "\n".join(record["unit_tests"]),
    ]
    code = "\n\n".join(part for part in code_parts if part).strip() + "\n"
    passed, stdout, stderr = run_python(code)
    metadata = {
        "id": record["id"],
        "domain": record["domain"],
        "passed": passed,
        "stdout": stdout,
        "stderr": stderr,
        "num_tests": len(record["unit_tests"]),
    }
    return passed, metadata


def main() -> None:
    ensure_standard_dirs()
    records = load_jsonl(INPUT_FILE)
    verified: list[dict] = []
    failures: list[dict] = []
    execution_results: list[dict] = []

    for record in records:
        passed, metadata = verify_math(record) if record["domain"] == "math" else verify_code(record)
        execution_results.append(metadata)
        enriched = dict(record)
        enriched["verification"] = metadata
        if passed:
            verified.append(enriched)
        else:
            failures.append(enriched)

    write_jsonl(verified, VERIFIED_FILE)
    write_jsonl(failures, FAILURE_FILE)
    write_jsonl(execution_results, RESULTS_FILE)
    print("✅ 程序执行与单元测试验证完成。")
    print(
        {
            "num_records": len(records),
            "verified_records": len(verified),
            "failed_records": len(failures),
        }
    )


if __name__ == "__main__":
    main()


### `src/package_textbook.py` - 打包教材产物

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict

from pipeline_utils import BOOKS_DIR, PROCESSED_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json, write_jsonl

INPUT_FILE = PROCESSED_DIR / "verified_textbook.jsonl"
PLAN_FILE = PROCESSED_DIR / "chapter_plan.json"
CURRICULUM_FILE = PROCESSED_DIR / "curriculum_map.json"
CATALOG_FILE = PROCESSED_DIR / "textbook_catalog.json"
STYLE_GUIDE_FILE = PROCESSED_DIR / "editorial_style_guide.md"
MATH_BOOK_FILE = BOOKS_DIR / "foundations_of_quantitative_reasoning.md"
CODE_BOOK_FILE = BOOKS_DIR / "python_problem_solving_workbook.md"
TEACHER_GUIDE_FILE = BOOKS_DIR / "teacher_guide.md"

MATH_TOPIC_SEQUENCE = [
    "arithmetic_word_problems",
    "fractions_and_proportions",
    "percentages_and_rates",
    "rates_and_motion",
    "geometry_and_measurement",
]
CODE_TOPIC_SEQUENCE = [
    "function_design",
    "lists_and_iteration",
    "string_algorithms",
    "search_and_sort",
    "graphs_and_trees",
]
DIFFICULTY_SEQUENCE = ["core", "advanced", "extension"]

MATH_NOTES = {
    "arithmetic_word_problems": "Start with quantity tracking before introducing more compressed symbolic reasoning.",
    "fractions_and_proportions": "Connect equal partitions to rate tables and unit scaling.",
    "percentages_and_rates": "Reinterpret percentages as multipliers and relative changes.",
    "rates_and_motion": "Use unit rates and repeated intervals to model time-dependent scenarios.",
    "geometry_and_measurement": "Apply formulas only after students identify which dimensions are known.",
}
CODE_NOTES = {
    "function_design": "Teach interface design before optimization so learners can read code contracts clearly.",
    "lists_and_iteration": "Develop traversal habits and accumulator patterns before introducing more compact idioms.",
    "string_algorithms": "Highlight local state updates, boundary checks, and repeated scans.",
    "search_and_sort": "Frame efficiency as a consequence of comparison strategy and loop invariants.",
    "graphs_and_trees": "Introduce recursive structure only after learners are comfortable with helper functions and state.",
}
STOPWORDS = {
    "the", "and", "with", "from", "into", "through", "there", "their", "about", "which", "would",
    "could", "should", "what", "when", "where", "while", "write", "function", "given", "using",
    "find", "calculate", "return", "after", "before", "into", "then", "than", "each", "have",
    "will", "your", "that", "this", "these", "those", "many", "much", "does", "make", "made",
}


def topic_order(domain: str, topic: str) -> int:
    sequence = MATH_TOPIC_SEQUENCE if domain == "math" else CODE_TOPIC_SEQUENCE
    return sequence.index(topic) if topic in sequence else len(sequence)


def difficulty_order(level: str) -> int:
    return DIFFICULTY_SEQUENCE.index(level) if level in DIFFICULTY_SEQUENCE else len(DIFFICULTY_SEQUENCE)


def chapter_learning_objectives(domain: str, topic: str) -> list[str]:
    if domain == "math":
        return [
            "Translate word conditions into explicit arithmetic steps.",
            "Track intermediate quantities without losing the target unknown.",
            "Check the final result against the story context.",
        ]
    return [
        "Read a function prompt and identify the required input-output contract.",
        "Use tests to confirm correctness on normal and edge cases.",
        "Explain why the chosen control flow matches the problem structure.",
    ]


def chapter_key_terms(domain: str, topic: str) -> list[str]:
    if domain == "math":
        if topic == "percentages_and_rates":
            return ["base quantity", "rate", "remaining value", "multiplier"]
        if topic == "rates_and_motion":
            return ["unit rate", "time interval", "total distance", "accumulation"]
        if topic == "fractions_and_proportions":
            return ["ratio", "fraction", "equivalent scaling", "part-whole relation"]
        return ["quantity", "step-by-step reasoning", "intermediate value", "final answer"]
    if topic == "string_algorithms":
        return ["scan", "index", "character test", "state update"]
    if topic == "lists_and_iteration":
        return ["loop", "accumulator", "transformation", "condition"]
    return ["function contract", "test case", "helper variable", "assertion"]


def chapter_common_mistakes(domain: str, topic: str) -> list[str]:
    if domain == "math":
        return [
            "Updating the wrong remaining quantity after an intermediate step.",
            "Mixing percent values with decimal multipliers.",
            "Stopping at an intermediate result instead of the requested final answer.",
        ]
    return [
        "Returning too early before all input elements are processed.",
        "Ignoring an edge case covered by the provided assertions.",
        "Writing code that works on one example but not on the full test set.",
    ]


def prerequisite_topics(domain: str, topic: str) -> list[str]:
    sequence = MATH_TOPIC_SEQUENCE if domain == "math" else CODE_TOPIC_SEQUENCE
    if topic not in sequence:
        return []
    index = sequence.index(topic)
    return sequence[max(0, index - 1):index]


def derive_focus_phrase(text: str) -> str:
    cleaned = "".join(char.lower() if char.isalpha() or char.isspace() else " " for char in text)
    tokens = [token for token in cleaned.split() if len(token) > 3 and token not in STOPWORDS]
    if not tokens:
        return "Core Applications"
    focus = " ".join(tokens[:3])
    return focus.title()


def format_chapter(record: dict) -> str:
    objectives = "\n".join(f"- {item}" for item in record["learning_objectives"])
    prerequisites = ", ".join(item.replace("_", " ").title() for item in record["prerequisites"]) or "None"
    key_terms = ", ".join(record["key_terms"])
    mistakes = "\n".join(f"- {item}" for item in record["common_mistakes"])
    assessment = "\n".join(f"- {item}" for item in record["end_of_chapter_checks"])

    sections = [
        f"## {record['chapter_code']} {record['chapter_title']}",
        "",
        f"Stage: {record['difficulty'].title()}",
        f"Topic: {record['topic'].replace('_', ' ').title()}",
        f"Prerequisites: {prerequisites}",
        "",
        "Learning objectives",
        objectives,
        "",
        "Why this chapter matters",
        record["chapter_rationale"],
        "",
        "Key terms",
        key_terms,
        "",
        "Lesson text",
        record["lesson_text"],
        "",
        "Worked example",
        f"Question: {record.get('worked_example_question', record['exercise_question'])}",
    ]

    if record["domain"] == "math":
        sections.extend(
            [
                f"Solution walkthrough: {record['worked_example_solution']}",
                f"Checkpoint exercise: {record['exercise_question']}",
                f"Verified answer: {record['exercise_answer']}",
            ]
        )
    else:
        sections.extend(
            [
                f"Programming exercise: {record['exercise_question']}",
                "Reference implementation:",
                "```python",
                record["reference_code"],
                "```",
                f"Verification coverage: {len(record['unit_tests'])} assertion(s).",
            ]
        )

    sections.extend(
        [
            "",
            "Common mistakes",
            mistakes,
            "",
            "End-of-chapter checks",
            assessment,
            "",
        ]
    )
    return "\n".join(sections)


def build_book(domain: str, title: str, subtitle: str, chapters: list[dict]) -> str:
    toc_lines = [f"- {record['chapter_code']} {record['chapter_title']}" for record in chapters]
    chapter_blocks = "\n\n".join(format_chapter(record) for record in chapters)
    return (
        f"# {title}\n\n"
        f"{subtitle}\n\n"
        "## Front Matter\n\n"
        "This volume is a reproducible synthetic textbook artifact generated from verified seed tasks. "
        "Each chapter includes objectives, prerequisite tags, a worked example, and end-of-chapter checks.\n\n"
        "## Table Of Contents\n\n"
        + "\n".join(toc_lines)
        + "\n\n## Main Text\n\n"
        + chapter_blocks
        + "\n"
    )


def main() -> None:
    ensure_standard_dirs()
    plan = load_json(PLAN_FILE)
    records = load_jsonl(INPUT_FILE)

    grouped: dict[str, list[dict]] = defaultdict(list)
    for record in records:
        grouped[record["domain"]].append(record)

    curriculum = {
        "volumes": [],
        "chapter_count": len(records),
        "domain_distribution": dict(Counter(record["domain"] for record in records)),
    }

    enriched_records: list[dict] = []
    catalog = {"books": []}
    book_specs = [
        ("math", "Volume I. Foundations of Quantitative Reasoning", "A compact workbook for arithmetic, rates, and proportional reasoning.", MATH_BOOK_FILE),
        ("code", "Volume II. Python Problem-Solving Workbook", "A structured beginner-to-intermediate programming text with verified examples.", CODE_BOOK_FILE),
    ]

    for volume_index, (domain, title, subtitle, output_path) in enumerate(book_specs, start=1):
        chapter_records = sorted(
            grouped.get(domain, []),
            key=lambda item: (topic_order(domain, item["topic"]), difficulty_order(item["difficulty"]), item["id"]),
        )
        for chapter_index, record in enumerate(chapter_records, start=1):
            topic_note = MATH_NOTES.get(record["topic"], "") if domain == "math" else CODE_NOTES.get(record["topic"], "")
            base_title = record["topic"].replace("_", " ").title()
            focus_phrase = derive_focus_phrase(record.get("exercise_question", record.get("worked_example_question", "")))
            record["volume"] = volume_index
            record["chapter_number"] = chapter_index
            record["chapter_code"] = f"Chapter {chapter_index:02d}"
            record["chapter_title"] = f"{base_title}: {focus_phrase}"
            record["prerequisites"] = prerequisite_topics(domain, record["topic"])
            record["learning_objectives"] = chapter_learning_objectives(domain, record["topic"])
            record["key_terms"] = chapter_key_terms(domain, record["topic"])
            record["common_mistakes"] = chapter_common_mistakes(domain, record["topic"])
            record["chapter_rationale"] = topic_note
            record["end_of_chapter_checks"] = [
                "State the main idea of the chapter in one sentence.",
                "Re-solve the checkpoint exercise without looking at the example.",
                "Explain one mistake that would lead to a wrong answer.",
            ]
            enriched_records.append(record)

        book_text = build_book(domain, title, subtitle, chapter_records)
        output_path.write_text(book_text, encoding="utf-8")
        catalog["books"].append(
            {
                "domain": domain,
                "title": title,
                "subtitle": subtitle,
                "path": output_path.relative_to(output_path.parent.parent).as_posix(),
                "num_chapters": len(chapter_records),
            }
        )
        curriculum["volumes"].append(
            {
                "volume": volume_index,
                "domain": domain,
                "title": title,
                "subtitle": subtitle,
                "num_chapters": len(chapter_records),
                "topics_in_order": [record["topic"] for record in chapter_records],
            }
        )

    style_guide = """# Editorial Style Guide

## Positioning

- Write as if this were a concise printed workbook for guided self-study.
- Keep tone instructional, calm, and explicit.
- Prefer short paragraphs, numbered steps, and verified outcomes.

## Chapter Structure

- Every chapter should declare prerequisites and learning objectives.
- Every worked example must end with a checked answer.
- Every code chapter should mention verification through assertions.
- Every chapter should include common mistakes and end-of-chapter checks.

## Difficulty Policy

- `core`: direct application with one main idea.
- `advanced`: multi-step reasoning with a hidden intermediate quantity.
- `extension`: asks learners to connect ideas across steps or abstractions.
"""
    STYLE_GUIDE_FILE.write_text(style_guide, encoding="utf-8")

    teacher_guide = (
        "# Teacher Guide\n\n"
        f"Seed count: {plan['seed_count']}\n\n"
        "Recommended usage:\n"
        "- Use the math volume for model lessons on quantitative reasoning.\n"
        "- Use the code volume for lab sessions centered on assertions and debugging.\n"
        "- Assign one checkpoint exercise and one end-of-chapter reflection for each lesson.\n"
    )
    TEACHER_GUIDE_FILE.write_text(teacher_guide, encoding="utf-8")

    write_json(curriculum, CURRICULUM_FILE)
    write_json(catalog, CATALOG_FILE)
    write_jsonl(enriched_records, INPUT_FILE)
    print("✅ 教材目录、先修关系和成册输出生成完成。")
    print(
        {
            "num_books": len(catalog["books"]),
            "num_chapters": len(enriched_records),
        }
    )


if __name__ == "__main__":
    main()


### `src/quality_control.py` - 质量控制

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, estimated_tokens, load_jsonl, normalize_text, write_json, write_jsonl

INPUT_FILE = PROCESSED_DIR / "verified_textbook.jsonl"
AUDIT_FILE = PROCESSED_DIR / "quality_audit.jsonl"
LOW_QUALITY_FILE = PROCESSED_DIR / "low_quality_flags.jsonl"
SUMMARY_FILE = PROCESSED_DIR / "quality_summary.json"
MANUAL_REVIEW_FILE = PROCESSED_DIR / "manual_review_samples.jsonl"


def validate_record(record: dict, seen_hashes: set[str]) -> tuple[dict, dict | None]:
    flags: list[str] = []
    if not record.get("chapter_title"):
        flags.append("missing_title")
    if not record.get("lesson_text"):
        flags.append("missing_lesson_text")
    if not record.get("chapter_markdown"):
        flags.append("missing_markdown")
    if not record.get("learning_objectives"):
        flags.append("missing_learning_objectives")
    if not record.get("end_of_chapter_checks"):
        flags.append("missing_end_of_chapter_checks")
    if record["domain"] == "math" and not record.get("exercise_answer"):
        flags.append("missing_math_answer")
    if record["domain"] == "code" and not record.get("unit_tests"):
        flags.append("missing_unit_tests")
    if not record.get("verification", {}).get("passed"):
        flags.append("verification_failed")
    if estimated_tokens(record.get("chapter_markdown", "")) < 60:
        flags.append("chapter_too_short")

    fingerprint = normalize_text(record.get("chapter_markdown", ""))
    if fingerprint in seen_hashes:
        flags.append("duplicate_chapter")
    else:
        seen_hashes.add(fingerprint)

    audit = {
        "id": record["id"],
        "domain": record["domain"],
        "topic": record["topic"],
        "difficulty": record["difficulty"],
        "passed": not flags,
        "flags": flags,
    }
    return audit, (audit if flags else None)


def main() -> None:
    records = load_jsonl(INPUT_FILE)
    seen_hashes: set[str] = set()
    audits: list[dict] = []
    low_quality: list[dict] = []

    for record in records:
        audit, maybe_low = validate_record(record, seen_hashes)
        audits.append(audit)
        if maybe_low:
            low_quality.append(maybe_low)

    summary = {
        "total_records": len(audits),
        "passed_records": sum(1 for audit in audits if audit["passed"]),
        "failed_records": len(low_quality),
        "domain_distribution": dict(Counter(record["domain"] for record in records)),
        "topic_distribution": dict(Counter(record["topic"] for record in records)),
        "difficulty_distribution": dict(Counter(record["difficulty"] for record in records)),
    }

    write_jsonl(audits, AUDIT_FILE)
    write_jsonl(low_quality, LOW_QUALITY_FILE)
    write_jsonl(records[:8], MANUAL_REVIEW_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ 质量风控与人工抽检样本生成完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/prepare_training_data.py` - 封装训练数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict

from pipeline_utils import (
    PROCESSED_DIR,
    TRAINING_DIR,
    deterministic_bucket,
    estimated_tokens,
    ensure_standard_dirs,
    load_jsonl,
    write_json,
    write_jsonl,
)

INPUT_FILE = PROCESSED_DIR / "verified_textbook.jsonl"
LOW_QUALITY_FILE = PROCESSED_DIR / "low_quality_flags.jsonl"
FINAL_FILE = TRAINING_DIR / "final_textbook_dataset.jsonl"
TRAIN_FILE = TRAINING_DIR / "train.jsonl"
VAL_FILE = TRAINING_DIR / "val.jsonl"
SMOKE_FILE = TRAINING_DIR / "smoke_test.jsonl"
MANIFEST_FILE = TRAINING_DIR / "training_manifest.json"


def make_training_record(record: dict) -> dict:
    instruction = (
        f"Write a {record['domain']} textbook section for the topic `{record['topic']}` "
        f"at `{record['difficulty']}` difficulty. Include explanation, exercise, and verified answer."
    )
    response = record["chapter_markdown"]
    return {
        "id": record["id"],
        "domain": record["domain"],
        "topic": record["topic"],
        "difficulty": record["difficulty"],
        "instruction": instruction,
        "response": response,
        "source_dataset": record["source_dataset"],
    }


def main() -> None:
    ensure_standard_dirs()
    blocked = {record["id"] for record in load_jsonl(LOW_QUALITY_FILE)}
    verified = [record for record in load_jsonl(INPUT_FILE) if record["id"] not in blocked]
    final_records = [make_training_record(record) for record in verified]
    final_records = sorted(final_records, key=lambda item: item["id"])

    train_records: list[dict] = []
    val_records: list[dict] = []
    smoke_by_domain: dict[str, list[dict]] = defaultdict(list)

    for record in final_records:
        if deterministic_bucket(record["id"]) < 85:
            train_records.append(record)
        else:
            val_records.append(record)
        if len(smoke_by_domain[record["domain"]]) < 6:
            smoke_by_domain[record["domain"]].append(record)

    smoke_records: list[dict] = []
    for domain in sorted(smoke_by_domain):
        smoke_records.extend(smoke_by_domain[domain])

    total_tokens = sum(
        estimated_tokens(record["instruction"] + "\n" + record["response"]) for record in final_records
    )

    manifest = {
        "num_records": len(final_records),
        "num_train_records": len(train_records),
        "num_val_records": len(val_records),
        "num_smoke_records": len(smoke_records),
        "domain_distribution": dict(Counter(record["domain"] for record in final_records)),
        "topic_distribution": dict(Counter(record["topic"] for record in final_records)),
        "difficulty_distribution": dict(Counter(record["difficulty"] for record in final_records)),
        "estimated_tokens_total": total_tokens,
    }

    write_jsonl(final_records, FINAL_FILE)
    write_jsonl(train_records, TRAIN_FILE)
    write_jsonl(val_records, VAL_FILE)
    write_jsonl(smoke_records, SMOKE_FILE)
    write_json(manifest, MANIFEST_FILE)
    print("✅ 训练前准备完成。")
    print(manifest)


if __name__ == "__main__":
    main()


### `src/evaluate_factory.py` - 评估教材工厂

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import BOOKS_DIR, PROCESSED_DIR, REPORTS_DIR, TRAINING_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p4_metrics.json"
REPORT_FILE = REPORTS_DIR / "p4_report.md"


def main() -> None:
    ensure_standard_dirs()
    seed_pool = load_jsonl(PROCESSED_DIR / "seed_pool.jsonl")
    synthesized = load_jsonl(PROCESSED_DIR / "synthetic_textbook_chapters.jsonl")
    verified = load_jsonl(PROCESSED_DIR / "verified_textbook.jsonl")
    failures = load_jsonl(PROCESSED_DIR / "verification_failures.jsonl")
    quality = load_json(PROCESSED_DIR / "quality_summary.json")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")
    curriculum = load_json(PROCESSED_DIR / "curriculum_map.json")
    catalog = load_json(PROCESSED_DIR / "textbook_catalog.json")

    metrics = {
        "seed_count": len(seed_pool),
        "synthesized_count": len(synthesized),
        "verified_count": len(verified),
        "verification_pass_rate": round(len(verified) / max(1, len(synthesized)), 4),
        "verification_failure_count": len(failures),
        "domain_distribution": dict(Counter(record["domain"] for record in verified)),
        "topic_distribution": dict(Counter(record["topic"] for record in verified)),
        "difficulty_distribution": dict(Counter(record["difficulty"] for record in verified)),
        "training_manifest": manifest,
        "estimated_external_generation_cost_usd": round(len(synthesized) * 0.06, 2),
        "estimated_manual_review_hours": round(len(verified) * 3 / 60, 2),
        "estimated_manual_review_cost_rmb": round(len(verified) * 3 * 100 / 60, 2),
        "quality_summary": quality,
        "num_books": len(catalog["books"]),
        "curriculum_volumes": curriculum["volumes"],
    }
    write_json(metrics, METRICS_FILE)

    report = f"""# P4 Synthetic Math and Code Textbook Report

## 1. 项目目标与范围

- 构建一个可复现的小型合成教材工厂，覆盖数学知识讲解与 Python 编程知识讲解。
- 当前读者边界：面向入门到进阶学习者，输出形式包括概念正文、例题、练习、解答与验证片段。

## 2. 种子池与章节模板

- 种子总数：{metrics["seed_count"]}
- 学科分布：{metrics["domain_distribution"]}
- 难度分布：{metrics["difficulty_distribution"]}
- 主题分布：{metrics["topic_distribution"]}

## 3. 生成、验证与纠错

- 合成章节数：{metrics["synthesized_count"]}
- 通过执行与单元测试验证的章节数：{metrics["verified_count"]}
- 验证通过率：{metrics["verification_pass_rate"]}
- 失败样本数：{metrics["verification_failure_count"]}
- 教材卷册数：{metrics["num_books"]}

## 4. 质量风控

- 质量审计总数：{quality["total_records"]}
- 审计通过数：{quality["passed_records"]}
- 低质量样本数：{quality["failed_records"]}
- 风控覆盖：缺标题、缺正文、缺学习目标、缺章末检查、缺单测、执行失败、过短章节、重复章节

## 5. 结果评测与成本

- 最终训练样本：{manifest["num_records"]}
- 训练集划分：train={manifest["num_train_records"]} val={manifest["num_val_records"]} smoke={manifest["num_smoke_records"]}
- 估算总 token 数：{manifest["estimated_tokens_total"]}
- 参考外部生成成本估算：{metrics["estimated_external_generation_cost_usd"]} USD
- 人工抽检工时估算：{metrics["estimated_manual_review_hours"]} 小时
- 人工抽检成本估算：{metrics["estimated_manual_review_cost_rmb"]} 元
- 成册产物：{", ".join(book["title"] for book in catalog["books"])}

## 6. 扩展方向

- 从小型教材章节扩展到题库库、讲义、课程脚本与助教反馈数据。
- 增加更严格的数学求解器和代码裁判器，构建自动纠错闭环。
- 用真实课程大纲替换当前的规则模板，升级章节顺序与先修关系设计。
"""

    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P4 报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p4_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from datetime import UTC, datetime

from pipeline_utils import BOOKS_DIR, PROCESSED_DIR, REPORTS_DIR, ROOT_DIR, TRAINING_DIR, load_json, load_jsonl

SCRIPTS = sorted((ROOT_DIR / "src").glob("*.py"))
REQUIRED_FILES = [
    PROCESSED_DIR / "seed_pool.jsonl",
    PROCESSED_DIR / "chapter_plan.json",
    PROCESSED_DIR / "synthetic_textbook_chapters.jsonl",
    PROCESSED_DIR / "verified_textbook.jsonl",
    PROCESSED_DIR / "verification_failures.jsonl",
    PROCESSED_DIR / "execution_results.jsonl",
    PROCESSED_DIR / "quality_audit.jsonl",
    PROCESSED_DIR / "low_quality_flags.jsonl",
    PROCESSED_DIR / "manual_review_samples.jsonl",
    PROCESSED_DIR / "curriculum_map.json",
    PROCESSED_DIR / "textbook_catalog.json",
    PROCESSED_DIR / "editorial_style_guide.md",
    TRAINING_DIR / "final_textbook_dataset.jsonl",
    TRAINING_DIR / "train.jsonl",
    TRAINING_DIR / "val.jsonl",
    TRAINING_DIR / "smoke_test.jsonl",
    TRAINING_DIR / "training_manifest.json",
    REPORTS_DIR / "p4_metrics.json",
    REPORTS_DIR / "p4_report.md",
    BOOKS_DIR / "foundations_of_quantitative_reasoning.md",
    BOOKS_DIR / "python_problem_solving_workbook.md",
    BOOKS_DIR / "teacher_guide.md",
]
RESULTS_FILE = REPORTS_DIR / "p4_test_results.json"
REPORT_FILE = REPORTS_DIR / "p4_test_report.md"


def run_command(command: list[str]) -> dict:
    completed = subprocess.run(command, capture_output=True, text=True)
    return {
        "command": command,
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


def main() -> None:
    command_checks = []
    py_compile = run_command([sys.executable, "-m", "py_compile", *[str(path) for path in SCRIPTS]])
    py_compile["name"] = "py_compile"
    command_checks.append(py_compile)

    evaluate = run_command([sys.executable, str(ROOT_DIR / "src" / "evaluate_factory.py")])
    evaluate["name"] = "evaluate_factory"
    command_checks.append(evaluate)

    verified = load_jsonl(PROCESSED_DIR / "verified_textbook.jsonl")
    failures = load_jsonl(PROCESSED_DIR / "verification_failures.jsonl")
    smoke_records = load_jsonl(TRAINING_DIR / "smoke_test.jsonl")
    train_records = load_jsonl(TRAINING_DIR / "train.jsonl")
    val_records = load_jsonl(TRAINING_DIR / "val.jsonl")
    manifest = load_json(TRAINING_DIR / "training_manifest.json")
    quality = load_json(PROCESSED_DIR / "quality_summary.json")
    curriculum = load_json(PROCESSED_DIR / "curriculum_map.json")

    dataset_checks = [
        {
            "name": "required_files_exist",
            "passed": all(path.exists() for path in REQUIRED_FILES),
            "details": {"missing_files": [str(path.relative_to(ROOT_DIR)) for path in REQUIRED_FILES if not path.exists()]},
        },
        {
            "name": "both_domains_present",
            "passed": {"math", "code"} <= {record["domain"] for record in verified},
            "details": {"domain_distribution": dict(Counter(record["domain"] for record in verified))},
        },
        {
            "name": "curriculum_has_two_volumes",
            "passed": len(curriculum["volumes"]) == 2,
            "details": {"volumes": curriculum["volumes"]},
        },
        {
            "name": "verification_pass_rate_reasonable",
            "passed": len(verified) >= 0.8 * (len(verified) + len(failures)),
            "details": {"verified": len(verified), "failed": len(failures)},
        },
        {
            "name": "train_val_no_overlap",
            "passed": not ({record["id"] for record in train_records} & {record["id"] for record in val_records}),
            "details": {"overlap": len({record["id"] for record in train_records} & {record["id"] for record in val_records})},
        },
        {
            "name": "smoke_covers_both_domains",
            "passed": {"math", "code"} <= {record["domain"] for record in smoke_records},
            "details": {"smoke_domain_distribution": dict(Counter(record["domain"] for record in smoke_records))},
        },
        {
            "name": "quality_has_no_failures",
            "passed": quality["failed_records"] == 0,
            "details": quality,
        },
        {
            "name": "manifest_matches_final_count",
            "passed": manifest["num_train_records"] + manifest["num_val_records"] == manifest["num_records"],
            "details": manifest,
        },
    ]

    results = {
        "timestamp_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "overall_passed": all(item["passed"] for item in command_checks) and all(item["passed"] for item in dataset_checks),
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks) + sum(item["passed"] for item in dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }

    RESULTS_FILE.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

    lines = [
        "# P4 Test Report",
        "",
        f"- Timestamp: {results['timestamp_utc']}",
        f"- Overall status: {'PASS' if results['overall_passed'] else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for check in command_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(check['command'])}`")
        if check["stdout"]:
            lines.append(f"  - Stdout: `{check['stdout'][:600]}`")
        if check["stderr"]:
            lines.append(f"  - Stderr: `{check['stderr'][:600]}`")

    lines.extend(["", "## Dataset Checks", ""])
    for check in dataset_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{json.dumps(check['details'], ensure_ascii=False)}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(results, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"
REPORTS_DIR = DATA_DIR / "reports"
BOOKS_DIR = ROOT_DIR / "books"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_standard_dirs() -> None:
    for path in [PROCESSED_DIR, TRAINING_DIR, REPORTS_DIR, BOOKS_DIR]:
        ensure_dir(path)


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records: list[dict], path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(data, path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def deterministic_bucket(key: str, buckets: int = 100) -> int:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % buckets


def estimated_tokens(text: str) -> int:
    compact = re.sub(r"\s+", " ", text).strip()
    return max(1, len(compact) // 4)


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def clean_gsm8k_solution(answer: str) -> str:
    text = re.sub(r"<<([^>]+)>>", r"\1", answer)
    text = text.replace("####", "Final answer:")
    return normalize_text(text)


def extract_final_answer(answer: str) -> str | None:
    match = re.search(r"####\s*(.+)$", answer, re.MULTILINE)
    if match:
        return match.group(1).strip()
    return None
